# COLING tables (LaTeX)

The tables of the paper, built from the same results as `Coling_eval_summary.ipynb` (through `coling_eval.py`, imported as `ce`) and from the judge study of `human_evaluation.ipynb` (`judge_ablation_results/summary.csv`, `human_eval_results/`). Run on ampere: the results, the judge's trial records and the training logs are there.

Every section builds one table as a DataFrame, passes its columns through **formatters** (section 0.1), prints the LaTeX and writes it to `tables/<name>.tex`, so the paper can use `\input{tables/<name>}`. The paper needs `booktabs` only (already in its preamble).

| Section | File | Label | Where it goes in the paper |
|---|---|---|---|
| 1 | `baselines.tex` | `tab:baselines` | 4.2 Comparison with baselines ("Table X reports the downstream and interpretability results") |
| 2 | `param_tests.tex` | `tab:param-tests` | 4.1 Comparison of parameter settings (the p-values quoted per paragraph) |
| 3 | `seeds.tex` | `tab:seeds` | 4.1 Random state ("INCLUDE TABLE HERE FOR TRANSPARENCY") |
| 4 | `iter_time.tex` | `tab:iter-time` | the seconds-per-iteration table already in section 4 |
| 5 | `grid.tex` | `tab:grid` | 3.5 "listed with their decomposition times in Table X", "TABLE WITH DECOMP TIMES" |
| 6 | `all_runs.tex` | `tab:all-runs` | appendix, "BIG RESULT TABLE" (also the bond 50 / 200 runs) |
| 7 | `intrusion.tex` | `tab:intrusion` | appendix: word intrusion per model with its 95% confidence interval and diversity |
| 8 | `judges.tex` | `tab:judges` | appendix, "APPENDIX TABLE JUDGE RESULTS" |
| 9 | `baseline_tests.tex` | `tab:baseline-tests` | appendix: the tests behind 4.2 |

Not built here: the tensor-size table (typed by hand in the paper) and the WiC pilot (`../eval.ipynb`, section 6).

## 0. Setup

The settings of `Coling_eval_summary.ipynb`, section 0: keep the two in step. `OURS_MAIN` are the runs of the baseline comparison, fixed in advance (`HEADLINE[300]` there).

In [14]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.utils.capture import capture_output


import coling_eval as ce

SERIES = 'latest'
MIN_ITERS = 500
EXCLUDE = []
APPENDIX = ['*_d200000']
SKIP = {'word_analogy'}
JUDGE_CSV = None

OURS_MAIN = ['tucker_3g_r300_scSoftPlus_ss0.025', 'tt_4g_r300_scSoftPlus_ss0.025']
VOCAB_MIN = 0.5  # a baseline shares our vocabulary if it knows at least this share of our words
OUT_DIR = ce.COLING_DIR / 'tables'
OUT_DIR.mkdir(exist_ok=True)

S = ce.load(SERIES, min_iters=MIN_ITERS, exclude=[*EXCLUDE, *APPENDIX], skip=SKIP, judge_csv=JUDGE_CSV)
B = ce.eu.BASE                   # the default settings
T, AVG, ACC = ce.tasks(), ce.AVG, 'judge_acc'
SCORES = [*T, AVG, ACC]          # the score columns of every results table
F = ce.foundations(S)            # per row: kind, data, latent_rank, words

sweep: sweep_2026-09-30T172843.json (running)
methods: methods_2026-09-24T183147.json (dry-run)
  tt_4g_r100_scSoftPlus_ss1: left out: no checkpoint before the walltime stop (model file at iteration 30)
  not at the table's iteration count, named <model>_<iterations reached>: tucker_4g_r100_countingLogEps_ss0.025_200, tucker_4g_r100_scSoftPlus_ss0.025_d20000_155, tucker_4g_r100_scSoftPlus_ss0.025_rs2_195, tucker_4g_r100_scSoftPlus_ss0.025_rs3_185, tt_4g_r100_scSoftPlus_ss0.1_300, tt_4g_r1000_scSoftPlus_ss0.025_305, tt_4g_r100_scSoftPlus_ss0.025_tt300_400, tt_5g_r100_scSoftPlus_ss0.1_245, tt_5g_r300_scSoftPlus_ss0.025_400, tt_6g_r100_scSoftPlus_ss0.1_100, tt_6g_r100_countingLogEps_ss0.025_415, tt_6g_r200_scSoftPlus_ss0.025_275, tt_6g_r300_scSoftPlus_ss0.025_205
kept 25 of our runs and 12 baselines on our words (series latest); 16 rows left out, 12 *_full rows apart
judge: summary_2026-10-02T110542.csv; trials of 36 judged rows


### 0.1 Formatters

A formatter turns one value into LaTeX (`NA` for a missing value). `cells(df, formatters)` applies one formatter per column and marks the best value per column; `latex_table` wraps the result in a `booktabs` float; `emit` prints it and writes the file.

In [15]:
NA = '--'
_ESC = {'\\': r'\textbackslash{}', '&': r'\&', '%': r'\%', '$': r'\$', '#': r'\#', '_': r'\_', '{': r'\{',
        '}': r'\}', '~': r'\textasciitilde{}', '^': r'\textasciicircum{}', '<': r'\textless{}',
        '>': r'\textgreater{}'}


def esc(text):
    """Plain text as LaTeX."""
    return ''.join(_ESC.get(ch, ch) for ch in str(text))


def _missing(v):
    return v is None or (not isinstance(v, str) and pd.isna(v))


def fmt_text(v):
    """Formatter: plain text, escaped."""
    return NA if _missing(v) else esc(v)


def raw(v):
    """Formatter: the value is LaTeX already."""
    return NA if _missing(v) else str(v)


def fmt_num(decimals=3, sign=False):
    """Formatter: a number with `decimals` places; sign: always signed, in maths so the minus is a minus."""
    def f(v):
        if _missing(v):
            return NA
        s = f'{float(v):+.{decimals}f}' if sign else f'{float(v):.{decimals}f}'
        if float(s) == 0:  # no -0.000
            s = s.replace('-', '+') if sign else s.lstrip('-')
        return f'${s}$' if sign else s
    return f


def fmt_int(v):
    """Formatter: an integer, thousands separated by a thin space."""
    return NA if _missing(v) else f'{int(round(float(v))):,}'.replace(',', r'\,')


def fmt_p(v):
    """Formatter: a p-value, three places; below 0.001 as < 0.001."""
    if _missing(v):
        return NA
    return r'$<$0.001' if float(v) < 0.001 else f'{float(v):.3f}'


def fmt_share(v):
    """Formatter: a fraction as a percentage (0.025 -> 2.5\\%)."""
    return NA if _missing(v) else f'{100 * float(v):g}\\%'


def fmt_thousands(v):
    """Formatter: a count in thousands (10000 -> 10k)."""
    return NA if _missing(v) else f'{float(v) / 1000:g}k'


def fmt_millions(v):
    """Formatter: a count in millions (2020000 -> 2.02M)."""
    if _missing(v):
        return NA
    x = float(v) / 1e6
    return (f'{x:.2f}'.rstrip('0').rstrip('.') if x < 100 else f'{x:,.0f}'.replace(',', r'\,')) + 'M'


def fmt_secs(v):
    """Formatter: seconds, two significant digits below 10."""
    if _missing(v):
        return NA
    v = float(v)
    return f'{v:.2f}' if v < 1 else f'{v:.1f}' if v < 10 else f'{v:.0f}'


def cells(df, formatters=None, default=fmt_text, best=(), second=(), bold=None):
    """df as a DataFrame of LaTeX strings: column -> formatter (`default` for the others).
    best: columns whose highest value is set bold; second: columns whose runner-up is underlined (values equal
    at the printed precision share the mark). bold: boolean DataFrame of further cells to set bold."""
    formatters = formatters or {}
    out = pd.DataFrame(index=df.index)
    for c in df.columns:
        f = formatters.get(c, default)
        s = [f(v) for v in df[c]]
        if c in best or c in second:
            vals = pd.to_numeric(df[c], errors='coerce').astype(float).to_numpy()
            ok = np.isfinite(vals)
            if ok.any():
                top = f(vals[ok].max())
                is_top = np.array([x == top for x in s]) & ok
                rest = ok & ~is_top
                runner = f(vals[rest].max()) if c in second and rest.any() else None
                s = [rf'\textbf{{{x}}}' if t and c in best else rf'\underline{{{x}}}' if o and x == runner else x
                     for x, t, o in zip(s, is_top, rest)]
        if bold is not None and c in bold.columns:
            s = [rf'\textbf{{{x}}}' if b else x for x, b in zip(s, bold[c].to_numpy(dtype=bool))]
        out[c] = s
    return out


def latex_table(body, caption, label, header=None, align=None, groups=None, sections=None, headings=True,
                wide=False, size=r'\small', tabcolsep=None, placement='t'):
    """A booktabs table float from a DataFrame of LaTeX strings (cells()).
    header: column -> LaTeX heading (default: the column name, escaped). align: the column spec (default: the
    first column l, the rest r). groups: [(heading, number of columns)], a row of spanning headings above the
    column headings ('' = none); a list of such lists gives several rows. sections: one label per row; a change
    of label starts a block with a rule and (headings) the label in italics. wide: table* (both columns)."""
    cols = list(body.columns)
    header = header or {}
    env = 'table*' if wide else 'table'
    lines = [rf'\begin{{{env}}}[{placement}]', r'  \centering', f'  {size}']
    if tabcolsep:
        lines.append(rf'  \setlength{{\tabcolsep}}{{{tabcolsep}}}')
    lines += [rf"  \begin{{tabular}}{{{align or 'l' + 'r' * (len(cols) - 1)}}}", r'    \toprule']
    for row in (groups if groups and isinstance(groups[0], list) else [groups] if groups else []):
        heads, rules, at = [], [], 1
        for title, n in row:
            heads.append(rf'\multicolumn{{{n}}}{{c}}{{{title}}}' if title else ' & '.join([''] * n))
            if title:
                rules.append(rf'\cmidrule(lr){{{at}-{at + n - 1}}}')
            at += n
        lines.append('    ' + ' & '.join(heads) + r' \\')
        if rules:
            lines.append('    ' + ' '.join(rules))
    lines += ['    ' + ' & '.join(header.get(c, esc(c)) for c in cols) + r' \\', r'    \midrule']
    sections = None if sections is None else list(sections)
    for i, row in enumerate(body.astype(str).to_numpy()):
        if sections is not None and (i == 0 or sections[i] != sections[i - 1]):
            if i:
                lines.append(r'    \midrule')
            if headings and sections[i]:
                lines.append(rf'    \multicolumn{{{len(cols)}}}{{l}}{{\textit{{{sections[i]}}}}} \\')
        lines.append('    ' + ' & '.join(row) + r' \\')
    lines += [r'    \bottomrule', r'  \end{tabular}', rf'  \caption{{{caption}}}', rf'  \label{{{label}}}',
              rf'\end{{{env}}}']
    return '\n'.join(lines)


TABLES = {}


def emit(name, latex):
    """Prints a table and writes it to tables/<name>.tex."""
    TABLES[name] = latex
    (OUT_DIR / f'{name}.tex').write_text(latex + '\n', encoding='utf-8')
    print(latex)

### 0.2 Names

How tasks, baselines and our runs are called in the tables. A baseline missing from `BASELINE` keeps its run name. A run of ours is named by its block (`Tucker-TT, 4-gram`) and by what it changes from the default settings (`extras`).

In [16]:
TASK = {'word_analogy': 'Analogy', 'sentiment': 'Sent.', 'TREC': 'TREC', 'discrim_attr': 'Attr.',
        'news_computer': 'Comp.', 'news_religion': 'Relig.', 'news_sports': 'Sports', 'np_bracketing': 'NP',
        'wordsim353': 'WS-353', AVG: 'Avg.', ACC: 'Intr.'}
TASK_LEGEND = (r'Sent.: sentiment classification; TREC: question classification; Attr.: discriminative attributes; '
               r'Comp., Relig., Sports: 20 Newsgroups; NP: noun-phrase bracketing; WS-353: word similarity '
               r"(Spearman's $\rho$; accuracy elsewhere); Avg.: mean over the downstream tasks; Intr.: "
               r'word-intrusion accuracy of the LLM judge (chance 0.20)')
SCORE_HEAD = {c: TASK.get(c, esc(c)) for c in SCORES}
SCORE_FMT = {c: fmt_num(2) for c in SCORES}

BASELINE = {'glove_100': 'GloVe', 'glove_300': 'GloVe', 'w2v': 'word2vec',
            'polar_glove_k500': 'POLAR (GloVe)', 'polar_w2v_k500': 'POLAR (word2vec)',
            'spine_glove': 'SPINE (GloVe)', 'spine_w2v': 'SPINE (word2vec)',
            'spowv_glove': 'SPOWV (GloVe)', 'spowv_w2v': 'SPOWV (word2vec)',
            'nnse_300': 'NNSE', 'nnse_1000': 'NNSE', 'sinr_bnc': 'SINr'}
SECTIONS = ['Dense', 'Post-hoc interpretable', 'Interpretable by construction']
FAMILY = {'tucker': 'Tucker', 'tt': 'Tucker-TT'}
ORDER = {'family': ['tucker', 'tt'], 'method': ['scSoftPlus', 'countingLog', 'countingLogEps']}
CONFIG_HEAD = {'r': '$r$', 'pop': 'Population', 'ss': 'Subs.', 'V': '$|V|$', 'bond': r'$\rho$', 'seed': 'Seed'}
CONFIG_FMT = {'r': fmt_int, 'pop': fmt_text, 'ss': fmt_share, 'V': fmt_thousands, 'bond': fmt_int, 'seed': fmt_int}


def baseline_section(kind):
    """The block of a baseline, from its kind in ce.foundations."""
    if kind == 'dense':
        return SECTIONS[0]
    return SECTIONS[2] if any(k in kind for k in ('NNSE', 'SINr')) else SECTIONS[1]


def baseline_rows(models):
    """The baselines `models` in table order: block, name and number of dimensions."""
    info = pd.DataFrame({'section': [baseline_section(F.at[m, 'kind']) for m in models],
                         'Model': [BASELINE.get(m, esc(m)) for m in models],
                         'Dim.': [F.at[m, 'latent_rank'] for m in models]}, index=list(models))
    rank = {s: i for i, s in enumerate(SECTIONS)}
    return info.sort_values(['section', 'Model', 'Dim.'], key=lambda s: s.map(rank) if s.name == 'section' else s)


def baseline_label(m):
    """A baseline's name, with its number of dimensions as a subscript where two baselines share the name."""
    name = BASELINE.get(m, esc(m))
    twins = [b for b, v in BASELINE.items() if v == name and b in F.index]
    if len(twins) > 1 and pd.notna(F.at[m, 'latent_rank']):
        return rf"{name}$_{{{int(F.at[m, 'latent_rank'])}}}$"
    return name


def run_title(r):
    """The block of a run of ours: Tucker-TT, 4-gram."""
    return f"{FAMILY.get(r['family'], esc(r['family']))}, {int(r['ngram'])}-gram"


def extras(r, skip=()):
    """What a run of ours changes from the default settings, as LaTeX ('' for the default run)."""
    out = []
    if 'rank' not in skip and int(r['rank']) != B['rank']:
        out.append(rf"$r{{=}}{int(r['rank'])}$")
    if 'method' not in skip and r['method'] != B['method']:
        out.append(esc(r['method']))
    if 'ss_frac' not in skip and not np.isclose(float(r['ss_frac']), B['ss_frac']):
        out.append(rf"{100 * float(r['ss_frac']):g}\% subsampling")
    if 'dims' not in skip and int(r['dims']) != B['dims']:
        out.append(f"{int(r['dims']) // 1000}k words")
    if 'tt_rank' not in skip and r['family'] == 'tt' and int(r['tt_rank']) != B['tt_rank']:
        out.append(rf"$\rho{{=}}{int(r['tt_rank'])}$")
    if 'random_state' not in skip and int(r['random_state']) != B['random_state']:
        out.append(f"seed {int(r['random_state'])}")
    return ', '.join(out)


def config_sort(df):
    """Our runs in table order: Tucker before Tucker-TT, then n-gram, rank, population, ..."""
    keys = ['family', 'ngram', 'rank', 'method', 'ss_frac', 'dims', 'tt_rank', 'random_state']
    return df.sort_values(keys, kind='stable', key=lambda s: (
        s.map({v: i for i, v in enumerate(ORDER[s.name])}).fillna(99) if s.name in ORDER else s.astype(float)))


def config_columns(df):
    """The settings of our runs as table columns (CONFIG_HEAD / CONFIG_FMT); bond only for Tucker-TT."""
    return pd.DataFrame({'r': df['rank'].astype(float), 'pop': df['method'], 'ss': df['ss_frac'].astype(float),
                         'V': df['dims'].astype(float),
                         'bond': df['tt_rank'].astype(float).where(df['family'].eq('tt'))}, index=df.index)

## 1. Main comparison: our two models against the baselines

Summary notebook: sections 1 and 3. Every baseline on our 10k words, then the runs of `OURS_MAIN`. Best per column bold, runner-up underlined.

In [17]:
for m in OURS_MAIN:
    if m not in S.df.index:
        print(f'{m}: not among the rows kept (see S.dropped)')
ours_main = [m for m in OURS_MAIN if m in S.df.index]
info = baseline_rows([m for m in S.df.index if S.df.at[m, 'family'] not in ce.OURS])
info = pd.concat([info, pd.DataFrame({'section': 'Ours', 'Model': [run_title(S.df.loc[m]) for m in ours_main],
                                      'Dim.': [float(S.df.at[m, 'rank']) for m in ours_main]}, index=ours_main)])
data = pd.concat([info[['Model', 'Dim.']], S.df.loc[info.index, SCORES].astype(float)], axis=1)
body = cells(data, {'Model': raw, 'Dim.': fmt_int, **SCORE_FMT}, best=SCORES, second=SCORES)
emit('baselines', latex_table(
    body, label='tab:baselines', header=SCORE_HEAD, sections=info['section'], wide=True, tabcolsep='4.5pt',
    groups=[('', 2), ('Downstream tasks', len(T) + 1), ('Interp.', 1)],
    caption=(r'Downstream and interpretability results of our two models and the baselines, all restricted to '
             r'our 10\,000-word vocabulary. ' + TASK_LEGEND + r'. Best score per column in bold, second best '
             r'underlined.')))

\begin{table*}[t]
  \centering
  \small
  \setlength{\tabcolsep}{4.5pt}
  \begin{tabular}{lrrrrrrrrrrr}
    \toprule
     &  & \multicolumn{9}{c}{Downstream tasks} & \multicolumn{1}{c}{Interp.} \\
    \cmidrule(lr){3-11} \cmidrule(lr){12-12}
    Model & Dim. & Sent. & TREC & Attr. & Comp. & Relig. & Sports & NP & WS-353 & Avg. & Intr. \\
    \midrule
    \multicolumn{12}{l}{\textit{Dense}} \\
    GloVe & 100 & 0.75 & 0.77 & 0.59 & 0.68 & \underline{0.83} & 0.82 & 0.74 & 0.50 & 0.71 & 0.44 \\
    GloVe & 300 & 0.76 & 0.80 & 0.62 & 0.75 & \textbf{0.84} & 0.86 & 0.74 & 0.54 & 0.74 & 0.36 \\
    word2vec & 300 & \textbf{0.78} & 0.82 & \textbf{0.65} & 0.74 & \textbf{0.84} & 0.86 & \textbf{0.78} & \textbf{0.68} & \textbf{0.77} & -- \\
    \midrule
    \multicolumn{12}{l}{\textit{Post-hoc interpretable}} \\
    POLAR (GloVe) & 500 & 0.76 & 0.79 & 0.60 & 0.66 & 0.80 & 0.80 & 0.73 & 0.58 & 0.71 & 0.56 \\
    POLAR (word2vec) & 500 & \underline{0.77} & 0.81 & 0.60 & 0.72 & \textbf{0.84} & 0.86 &

## 2. Effect of each setting

Summary notebook: section 2 (`ce.analyse` per factor, the rows of `ce.summary`). The subsampling rows marked with a dagger come from the `same` series (each matched set at the checkpoint all its runs have), downstream tasks only; they are skipped if no `same_*` sweep is on disk.

In [18]:
FACTORS = ['family', 'tt_rank', 'ss_frac', 'method', 'ngram', 'rank', 'dims']
FACTOR = {'family': 'Decomposition', 'tt_rank': r'TT bond $\rho$', 'ss_frac': 'Subsampling',
          'method': 'Population', 'ngram': r'$n$-gram order', 'rank': 'Rank $r$', 'dims': 'Vocabulary'}


def level_tex(factor, token):
    """One level as ce._lv writes it (r100, 4g, b50, ss0.025, tucker) as LaTeX."""
    token = token.strip()
    if factor == 'family':
        return FAMILY.get(token, esc(token))
    if factor in ('tt_rank', 'rank'):
        return token[1:]
    if factor == 'ss_frac':
        return rf'{100 * float(token[2:]):g}\%'
    if factor == 'ngram':
        return token[:-1]
    return esc(token)


def levels_tex(factor, test, levels):
    """The levels of a test (a → b, or a < b < c for a trend) as LaTeX."""
    if ' → ' in levels:
        return r' $\rightarrow$ '.join(level_tex(factor, t) for t in levels.split(' → '))
    text = ' $<$ '.join(level_tex(factor, t) for t in levels.split(' < '))
    return f'trend: {text}' if test == 'trend' else text


R = {f: ce.analyse(S, f) for f in FACTORS}
try:
    with capture_output():
        S_same = ce.load('same', exclude=[*EXCLUDE, *APPENDIX], skip=SKIP, judge_csv=JUDGE_CSV)
        same = ce.analyse(S_same, 'ss_frac').tests
except Exception as e:  # no same_* sweep on disk
    print(f'no rows of the same series: {e}')
    same = pd.DataFrame()

frames = []
for f in FACTORS:
    if len(R[f].tests):
        frames.append(R[f].tests.assign(setting=FACTOR[f]))
    if f == 'ss_frac' and len(same):
        frames.append(same.assign(setting=FACTOR[f] + r'$^\dagger$'))
tests = pd.concat(frames, ignore_index=True)

data = pd.DataFrame({
    'Setting': tests['setting'].where(tests['setting'].ne(tests['setting'].shift()), ''),
    'Comparison': [levels_tex(f, t, lv) for f, t, lv in zip(tests['factor'], tests['test'], tests['levels'])],
    'Sets': tests['POLAR sets'],
    'pol_e': tests['POLAR effect'], 'pol_p': tests['POLAR p (Holm)'],
    'acc_p': tests['judge acc p (Holm)']})
significant = pd.DataFrame({'pol_p': tests['POLAR p (Holm)'].lt(ce.ALPHA),
                            'acc_p': tests['judge acc p (Holm)'].lt(ce.ALPHA)})
body = cells(data, {'Setting': raw, 'Comparison': raw, 'Sets': fmt_int,
                    'pol_e': fmt_num(2, sign=True), 'pol_p': fmt_p, 'acc_p': fmt_p}, bold=significant)
emit('param_tests', latex_table(
    body, label='tab:param-tests', align='llrrrr', sections=tests['setting'], headings=False, wide=True,
    header={'pol_e': 'Effect', 'pol_p': '$p$', 'acc_p': '$p$'},
    groups=[('', 3), (f'Downstream ({len(T)} tasks)', 2), ('Word intrusion', 1)],
    caption=(r'Effect of each setting, on matched sets of runs that differ in that setting only (Sets: their '
             r'number). Downstream tests take the tasks as units: Wilcoxon signed-rank for two levels (Effect: '
             r"rank-biserial $r$), Page's $L$ for a trend (Effect: mean Spearman $\rho$); a "
             r"positive Effect means that the later level scores higher. Word-intrusion tests "
             r"take the judge's trials as units: Cochran--Mantel--Haenszel for two levels, Cochran--Armitage for "
             r'a trend, stratified by set. All $p$ two-sided, Holm-corrected per setting and suite; bold: '
             r'$p < 0.05$. $^\dagger$: every run of a set at the last iteration they all reached.')))

\begin{table*}[t]
  \centering
  \small
  \begin{tabular}{llrrrr}
    \toprule
     &  &  & \multicolumn{2}{c}{Downstream (8 tasks)} & \multicolumn{1}{c}{Word intrusion} \\
    \cmidrule(lr){4-5} \cmidrule(lr){6-6}
    Setting & Comparison & Sets & Effect & $p$ & $p$ \\
    \midrule
    Decomposition & Tucker $\rightarrow$ Tucker-TT & 2 & $+0.56$ & 0.195 & 1.000 \\
    \midrule
    TT bond $\rho$ & trend: 50 $<$ 100 $<$ 200 & 1 & $+0.25$ & 0.777 & 1.000 \\
     & 50 $\rightarrow$ 100 & 1 & $+0.83$ & 0.117 & 1.000 \\
     & 100 $\rightarrow$ 200 & 1 & $-0.33$ & 0.777 & 1.000 \\
    \midrule
    Subsampling & trend: 2.5\% $<$ 10\% $<$ 100\% & 1 & $-0.31$ & 0.799 & 1.000 \\
     & 2.5\% $\rightarrow$ 10\% & 1 & $-0.11$ & 1.000 & 1.000 \\
     & 10\% $\rightarrow$ 100\% & 1 & $-0.11$ & 1.000 & 1.000 \\
    \midrule
    Subsampling$^\dagger$ & trend: 2.5\% $<$ 10\% $<$ 100\% & 1 & $-0.31$ & 0.799 & -- \\
     & 2.5\% $\rightarrow$ 10\% & 4 & $-0.22$ & 1.000 & -- \\
     & 10\% $\rightarrow$

## 3. Random seed

Summary notebook: section 2.8 (`ce.seed_noise`). The scores per seed, then the standard deviation over the seeds, the standard deviation over our seed-1 runs on our words, and their ratio.

In [19]:
with capture_output():
    S_seed = ce.with_seeds(S)
    seed_sets = ce.matched_sets(S_seed, 'random_state')
if not seed_sets:
    print('no seed replicates among the runs kept (check S.dropped and MIN_ITERS)')
else:
    R_seed = ce.Factor('random_state', seed_sets,
                       ce._order('random_state', [v for s in seed_sets.values() for v in s.index]),
                       [], None, pd.DataFrame())
    noise = ce.seed_noise(S_seed, R_seed)
    blocks, sections = [], []
    for s in seed_sets.values():
        first = S_seed.df.loc[s.iloc[0]]
        runs = S_seed.df.loc[s.to_numpy(), SCORES].astype(float)
        runs.insert(0, 'Run', [f'Seed {int(v)}' for v in s.index])
        blocks.append(cells(runs, {'Run': raw, **SCORE_FMT}))
        sections += [', '.join(p for p in (run_title(first), extras(first, skip=('random_state',))) if p)] * len(runs)
    for name, col, f in [('SD over seeds', 'pooled SD', fmt_num(3)), ('SD over settings', 'SD over settings', fmt_num(3)),
                         ('Ratio', 'seed / settings', lambda v: NA if _missing(v) else f'{100 * float(v):.0f}\\%')]:
        row = pd.DataFrame([noise[col].reindex(SCORES)], index=[name])
        row.insert(0, 'Run', name)
        blocks.append(cells(row, {'Run': raw}, default=f))
        sections.append('')
    emit('seeds', latex_table(
        pd.concat(blocks), label='tab:seeds', header={'Run': '', **SCORE_HEAD}, sections=sections, wide=True,
        groups=[('', 1), ('Downstream tasks', len(T) + 1), ('Interp.', 1)],
        caption=(r'Effect of the random seed (initialisation and subsample windows) on the default model. SD over '
                 r'seeds: standard deviation over the seeds; SD over settings: standard deviation over all our '
                 r'seed-1 runs on the 10\,000-word vocabulary; Ratio: the former as a share of the latter. '
                 + TASK_LEGEND + '.')))

\begin{table*}[t]
  \centering
  \small
  \begin{tabular}{lrrrrrrrrrr}
    \toprule
     & \multicolumn{9}{c}{Downstream tasks} & \multicolumn{1}{c}{Interp.} \\
    \cmidrule(lr){2-10} \cmidrule(lr){11-11}
     & Sent. & TREC & Attr. & Comp. & Relig. & Sports & NP & WS-353 & Avg. & Intr. \\
    \midrule
    \multicolumn{11}{l}{\textit{Tucker-TT, 4-gram}} \\
    Seed 1 & 0.67 & 0.91 & 0.53 & 0.60 & 0.73 & 0.63 & 0.77 & 0.48 & 0.66 & 0.83 \\
    Seed 2 & 0.67 & 0.88 & 0.50 & 0.62 & 0.73 & 0.65 & 0.77 & 0.53 & 0.67 & 0.84 \\
    Seed 3 & 0.66 & 0.89 & 0.54 & 0.60 & 0.67 & 0.63 & 0.77 & 0.43 & 0.65 & 0.89 \\
    \midrule
    SD over seeds & 0.006 & 0.014 & 0.020 & 0.010 & 0.035 & 0.009 & 0.003 & 0.049 & 0.011 & 0.032 \\
    SD over settings & 0.020 & 0.040 & 0.020 & 0.027 & 0.032 & 0.022 & 0.010 & 0.057 & 0.021 & 0.031 \\
    Ratio & 30\% & 34\% & 99\% & 35\% & 109\% & 41\% & 36\% & 85\% & 52\% & 103\% \\
    \bottomrule
  \end{tabular}
  \caption{Effect of the random seed (initialisation 

## 4. Seconds per iteration, and 5. the grid of runs

Summary notebook: section 4. Seconds per iteration are the median of the iteration times in each run's log (`ce.iter_times`), for every run of ours loaded, also those the cuts of section 0 left out. `hours` = seconds per iteration × iterations reached: the update time, without the in-loop evaluation and checkpointing. Core parameters: $r^n$ for Tucker, $2r\rho + (n-2)r\rho^2$ for the hybrid.

Table 4 keeps the default run of each (model, n-gram) column; table 5 lists every run.

In [20]:
L = S.loaded[S.loaded['family'].isin(ce.OURS)]
G = L[['family', 'method']].copy()
for c in ('ngram', 'rank', 'ss_frac', 'dims', 'tt_rank', 'random_state', 'reached'):
    G[c] = L[c].astype(float)
times = [ce.iter_times(S.model_files[run]) if run in S.model_files else [] for run in L['run']]
G['secs'] = [float(np.median(t)) if len(t) else np.nan for t in times]
G['hours'] = G['secs'] * G['reached'] / 3600
G['core'] = np.where(G['family'].eq('tt'), 2 * G['rank'] * G['tt_rank'] + (G['ngram'] - 2) * G['rank'] * G['tt_rank'] ** 2,
                     G['rank'] ** G['ngram'])
why = dict(zip(S.dropped.index, S.dropped['reason']))
G['left out'] = [why.get(m, '') for m in G.index]
G = config_sort(G)

default = G[G['rank'].eq(B['rank']) & G['method'].eq(B['method']) & np.isclose(G['ss_frac'], B['ss_frac'])
            & G['dims'].eq(B['dims']) & G['tt_rank'].isin([0, B['tt_rank']]) & G['random_state'].eq(B['random_state'])]
piv = default.pivot_table(index='family', columns='ngram', values='secs', aggfunc='first').reindex(ORDER['family'])
data = piv.copy()
data.insert(0, 'Model', [FAMILY[f] for f in piv.index])
body = cells(data, {'Model': raw}, default=fmt_secs)
emit('iter_time', latex_table(
    body, label='tab:iter-time', align='l' + 'c' * len(piv.columns),
    header={'Model': '', **{n: rf'$n={int(n)}$' for n in piv.columns}},
    caption=(r'Seconds per iteration at the default settings ($r=100$). Tucker is infeasible for $n\geq5$ (core '
             r'of 40\,GB and 4\,TB at $r=100$); the TT hybrid does not reduce parameters for $n\leq3$. '
             r'Table~\ref{tab:grid} lists every run.')))

\begin{table}[t]
  \centering
  \small
  \begin{tabular}{lcccc}
    \toprule
     & $n=3$ & $n=4$ & $n=5$ & $n=6$ \\
    \midrule
    Tucker & 0.38 & 75 & -- & -- \\
    Tucker-TT & -- & 11 & 22 & 33 \\
    \bottomrule
  \end{tabular}
  \caption{Seconds per iteration at the default settings ($r=100$). Tucker is infeasible for $n\geq5$ (core of 40\,GB and 4\,TB at $r=100$); the TT hybrid does not reduce parameters for $n\leq3$. Table~\ref{tab:grid} lists every run.}
  \label{tab:iter-time}
\end{table}


In [21]:
data = config_columns(G)
data['seed'] = G['random_state']
stopped = G['left out'].str.startswith('reached')
data['iters'] = [fmt_int(v) + (r'$^\dagger$' if cut else '') for v, cut in zip(G['reached'], stopped)]
for c in ('secs', 'hours', 'core'):
    data[c] = G[c]
body = cells(data, {**CONFIG_FMT, 'iters': raw, 'secs': fmt_secs, 'hours': fmt_num(1), 'core': fmt_millions})
emit('grid', latex_table(
    body, label='tab:grid', align='rl' + 'r' * (len(data.columns) - 2), wide=True,
    sections=[run_title(r) for _, r in G.iterrows()],
    header={**CONFIG_HEAD, 'iters': 'Iterations', 'secs': 's / iteration', 'hours': 'Hours', 'core': 'Core'},
    caption=(r'The runs of our grid and their cost. $r$: rank; Subs.: subsampling fraction; $|V|$: vocabulary '
             r'size; $\rho$: TT bond; s / iteration: median time of one update; Hours: that time over all '
             r'iterations, without the in-loop evaluation; Core: number of core parameters. $^\dagger$: stopped '
             r'at its walltime before 500 iterations and left out of the downstream evaluation.')))

\begin{table*}[t]
  \centering
  \small
  \begin{tabular}{rlrrrrrrrr}
    \toprule
    $r$ & Population & Subs. & $|V|$ & $\rho$ & Seed & Iterations & s / iteration & Hours & Core \\
    \midrule
    \multicolumn{10}{l}{\textit{Tucker, 3-gram}} \\
    100 & scSoftPlus & 2.5\% & 10k & -- & 1 & 500 & 0.38 & 0.1 & 1M \\
    100 & scSoftPlus & 10\% & 10k & -- & 1 & 500 & 1.2 & 0.2 & 1M \\
    100 & scSoftPlus & 100\% & 10k & -- & 1 & 500 & 7.8 & 1.1 & 1M \\
    100 & countingLog & 2.5\% & 10k & -- & 1 & 500 & 0.17 & 0.0 & 1M \\
    100 & countingLogEps & 2.5\% & 10k & -- & 1 & 500 & 0.38 & 0.1 & 1M \\
    200 & scSoftPlus & 2.5\% & 10k & -- & 1 & 500 & 1.2 & 0.2 & 8M \\
    300 & scSoftPlus & 2.5\% & 10k & -- & 1 & 500 & 3.0 & 0.4 & 27M \\
    \midrule
    \multicolumn{10}{l}{\textit{Tucker, 4-gram}} \\
    100 & scSoftPlus & 2.5\% & 10k & -- & 1 & 500 & 75 & 10.5 & 100M \\
    100 & scSoftPlus & 2.5\% & 10k & -- & 2 & 195$^\dagger$ & 69 & 3.8 & 100M \\
    100 & scSoftPlus & 2.5\% & 10k &

## 6. Every run of ours (appendix)

Summary notebook: section 1 (`ce.overview`), our rows only: the runs kept, on the downstream tasks and word intrusion. Best per column bold.

In [22]:
ours = config_sort(ce._ours(S.df))
data = pd.concat([config_columns(ours), ours[SCORES].astype(float)], axis=1)
body = cells(data, {**CONFIG_FMT, **SCORE_FMT}, best=SCORES)
emit('all_runs', latex_table(
    body, label='tab:all-runs', align='rl' + 'r' * (len(data.columns) - 2), header={**CONFIG_HEAD, **SCORE_HEAD},
    sections=[run_title(r) for _, r in ours.iterrows()], wide=True, size=r'\footnotesize', tabcolsep='4pt',
    groups=[('Settings', 5), ('Downstream tasks', len(T) + 1), ('Interp.', 1)],
    caption=(r'Results of every run of ours that completed 500 iterations. $r$: rank; Subs.: subsampling '
             r'fraction; $|V|$: vocabulary size; $\rho$: TT bond. ' + TASK_LEGEND + r'. Best score per column '
             r'in bold.')))

\begin{table*}[t]
  \centering
  \footnotesize
  \setlength{\tabcolsep}{4pt}
  \begin{tabular}{rlrrrrrrrrrrrrr}
    \toprule
    \multicolumn{5}{c}{Settings} & \multicolumn{9}{c}{Downstream tasks} & \multicolumn{1}{c}{Interp.} \\
    \cmidrule(lr){1-5} \cmidrule(lr){6-14} \cmidrule(lr){15-15}
    $r$ & Population & Subs. & $|V|$ & $\rho$ & Sent. & TREC & Attr. & Comp. & Relig. & Sports & NP & WS-353 & Avg. & Intr. \\
    \midrule
    \multicolumn{15}{l}{\textit{Tucker, 3-gram}} \\
    100 & scSoftPlus & 2.5\% & 10k & -- & 0.69 & 0.90 & 0.53 & 0.60 & 0.75 & 0.63 & \textbf{0.78} & 0.47 & 0.67 & 0.90 \\
    100 & scSoftPlus & 10\% & 10k & -- & 0.68 & 0.90 & 0.53 & 0.59 & 0.75 & 0.62 & \textbf{0.78} & 0.47 & 0.67 & 0.88 \\
    100 & scSoftPlus & 100\% & 10k & -- & 0.68 & 0.89 & 0.53 & 0.60 & 0.72 & 0.63 & \textbf{0.78} & 0.47 & 0.66 & 0.88 \\
    100 & countingLog & 2.5\% & 10k & -- & 0.66 & 0.71 & 0.52 & 0.58 & 0.70 & 0.61 & 0.77 & 0.37 & 0.62 & 0.91 \\
    100 & countingLogEps & 2.5\% & 

## 7. Word intrusion per model (appendix)

Summary notebook: section 1.1 (`ce.intrusion_table`): the judge's accuracy with its 95% Wilson confidence interval over the trials, and the diversity of the top words (the one place it is shown).

In [23]:
it = ce.intrusion_table(S).data
base = baseline_rows([m for m in it.index if S.df.at[m, 'family'] not in ce.OURS])
mine = config_sort(S.df.loc[[m for m in it.index if S.df.at[m, 'family'] in ce.OURS]])
info = pd.concat([base, pd.DataFrame({'section': ['Ours: ' + run_title(r) for _, r in mine.iterrows()],
                                      'Model': [extras(r) or 'default settings' for _, r in mine.iterrows()],
                                      'Dim.': mine['rank'].astype(float)}, index=mine.index)])
data = info[['Model', 'Dim.']].copy()
data['acc'] = it.loc[info.index, 'judge_acc'].astype(float)
data['ci'] = [f'{lo:.2f}--{hi:.2f}' for lo, hi in zip(it.loc[info.index, '95% CI low'], it.loc[info.index, '95% CI high'])]
data['div'] = it.loc[info.index, 'diversity'].astype(float)
body = cells(data, {'Model': raw, 'Dim.': fmt_int, 'acc': fmt_num(2), 'ci': raw, 'div': fmt_num(2)}, best=['acc'])
emit('intrusion', latex_table(
    body, label='tab:intrusion', sections=info['section'], size=r'\footnotesize', tabcolsep='4pt',
    header={'acc': 'Accuracy', 'ci': '95\\% CI', 'div': 'Diversity'},
    caption=(r'Word intrusion per model: accuracy of the LLM judge (chance 0.20) with its 95\% Wilson confidence '
             r'interval over the trials (one per dimension), and the diversity of the top words (distinct top '
             r'words over dimensions $\times$ 4). Our runs are named by what they change from the default '
             r'settings.')))

\begin{table}[t]
  \centering
  \footnotesize
  \setlength{\tabcolsep}{4pt}
  \begin{tabular}{lrrrr}
    \toprule
    Model & Dim. & Accuracy & 95\% CI & Diversity \\
    \midrule
    \multicolumn{5}{l}{\textit{Dense}} \\
    GloVe & 100 & 0.44 & 0.35--0.54 & 0.93 \\
    GloVe & 300 & 0.36 & 0.31--0.42 & 0.83 \\
    \midrule
    \multicolumn{5}{l}{\textit{Post-hoc interpretable}} \\
    POLAR (GloVe) & 500 & 0.56 & 0.52--0.60 & 0.92 \\
    POLAR (word2vec) & 500 & 0.55 & 0.50--0.59 & 0.92 \\
    SPINE (GloVe) & 1\,000 & 0.73 & 0.70--0.76 & 0.69 \\
    SPINE (word2vec) & 1\,000 & 0.80 & 0.77--0.82 & 0.74 \\
    SPOWV (GloVe) & 1\,000 & 0.39 & 0.36--0.42 & 0.68 \\
    SPOWV (word2vec) & 1\,000 & 0.47 & 0.44--0.50 & 0.68 \\
    \midrule
    \multicolumn{5}{l}{\textit{Interpretable by construction}} \\
    NNSE & 300 & 0.83 & 0.78--0.87 & 0.98 \\
    NNSE & 1\,000 & 0.87 & 0.85--0.89 & 0.92 \\
    SINr & 1\,431 & 0.48 & 0.45--0.50 & 0.57 \\
    \midrule
    \multicolumn{5}{l}{\textit{Ours:

## 8. Judge selection (appendix)

`human_evaluation.ipynb`, "Judge as one more annotator": `judge_ablation_results/summary.csv`, one row per (judge model, prompt), sorted by the φ ratio. The first row holds the human annotators: their mean φ and pick agreement with each other, and their mean accuracy per trial set (read from `human_eval_results/`, finished sessions only; a skipped trial counts as wrong). `JUDGE_PROMPTS` picks the prompts shown (None: all of them; the paper describes `default` only).

In [24]:
JUDGE_PROMPTS = None  # e.g. ['default']
SETS = {'4-gram_base': 'Tucker', '4-gram_tt': 'Tucker-TT'}  # the human trial sets: both default 4-gram models

J = pd.read_csv(ce.COLING_DIR / 'judge_ablation_results' / 'summary.csv')
if JUDGE_PROMPTS:
    J = J[J['prompt'].isin(JUDGE_PROMPTS)]
J = J.sort_values('phi_ratio', ascending=False).reset_index(drop=True)
acc_cols = [c for c in J.columns if c.startswith('judge_acc ')]


def human_accuracy(set_name):
    """Mean accuracy of the annotators who finished the trial set `set_name`."""
    accs = []
    for path in sorted((ce.COLING_DIR / 'human_eval_results').glob(f'{set_name}_w*.jsonl')):
        meta, responses = None, {}
        for line in path.read_text(encoding='utf-8').splitlines():
            if not line.strip():
                continue
            rec = json.loads(line)
            if rec.get('type') == 'meta':
                meta = rec
            elif rec.get('type') == 'response':
                responses[rec['dim']] = rec
            elif rec.get('type') == 'undo':
                responses.pop(rec['dim'], None)
        if meta and meta.get('model') == set_name and len(responses) == len(meta['tasks']):
            accs.append(sum(bool(r['correct']) for r in responses.values()) / len(responses))
    print(f'{set_name}: {len(accs)} finished human sessions')
    return float(np.mean(accs)) if accs else np.nan


judges = pd.DataFrame({'Judge': [m.split('/')[-1] for m in J['model']], 'Prompt': J['prompt'],
                       'phi': J['judge_phi'], 'phi_ratio': J['phi_ratio'], 'pick': J['judge_pick_agreement'],
                       'pick_ratio': J['pick_agreement_ratio'], **{c: J[c] for c in acc_cols}})
humans = pd.DataFrame([{'Judge': f"Human annotators ({int(J['n_humans'].iloc[0])})", 'Prompt': '',
                        'phi': J['human_phi'].iloc[0], 'phi_ratio': 1.0, 'pick': J['human_pick_agreement'].iloc[0],
                        'pick_ratio': 1.0, **{c: human_accuracy(c.split(' ', 1)[1]) for c in acc_cols}}])
numeric = [c for c in judges.columns if c not in ('Judge', 'Prompt')]
fmt = {'Judge': fmt_text, 'Prompt': fmt_text, **{c: fmt_num(2) for c in numeric}, **{c: fmt_num(2) for c in acc_cols}}
body = pd.concat([cells(humans, fmt), cells(judges, fmt, best=numeric)], ignore_index=True)
emit('judges', latex_table(
    body, label='tab:judges', align='ll' + 'r' * len(numeric), sections=['humans'] + ['judges'] * len(judges),
    headings=False, wide=True,
    header={'phi': r'$\phi$', 'phi_ratio': r'$\phi_j / \phi_h$', 'pick': 'Agreement', 'pick_ratio': 'Ratio',
            **{c: SETS.get(c.split(' ', 1)[1], esc(c.split(' ', 1)[1])) for c in acc_cols}},
    groups=[('', 2), ('Correctness', 2), ('Picked word', 2), ('Accuracy', len(acc_cols))],
    caption=(r'Judge selection on the trial sets annotated by three humans (the two default 4-gram models, 100 '
             r'dimensions each). $\phi$: mean correlation of the per-trial correctness with each human annotator '
             r'(for the humans: with the other annotators); Agreement: mean share of trials with the same word '
             r'picked as a human; $\phi_j / \phi_h$ and Ratio: relative to the human--human value. Accuracy: share '
             r'of intruders found per trial set. Sorted by $\phi_j / \phi_h$; best judge per column in bold.')))

4-gram_base: 3 finished human sessions
4-gram_tt: 3 finished human sessions
\begin{table*}[t]
  \centering
  \small
  \begin{tabular}{llrrrrrr}
    \toprule
     &  & \multicolumn{2}{c}{Correctness} & \multicolumn{2}{c}{Picked word} & \multicolumn{2}{c}{Accuracy} \\
    \cmidrule(lr){3-4} \cmidrule(lr){5-6} \cmidrule(lr){7-8}
    Judge & Prompt & $\phi$ & $\phi_j / \phi_h$ & Agreement & Ratio & Tucker & Tucker-TT \\
    \midrule
    Human annotators (3) &  & 0.42 & 1.00 & 0.88 & 1.00 & 0.86 & 0.89 \\
    \midrule
    Qwen3.5-9B & default & \textbf{0.33} & \textbf{0.79} & \textbf{0.83} & \textbf{0.95} & \textbf{0.82} & \textbf{0.88} \\
    Qwen3.5-9B & few-shot & \textbf{0.33} & 0.78 & 0.79 & 0.90 & 0.78 & 0.81 \\
    Qwen3.5-4B & default & 0.31 & 0.73 & 0.80 & 0.91 & \textbf{0.82} & 0.84 \\
    Llama-3.1-8B-Instruct & few-shot & 0.27 & 0.65 & 0.70 & 0.80 & 0.69 & 0.71 \\
    Qwen3.5-4B & few-shot & 0.26 & 0.61 & 0.72 & 0.82 & 0.74 & 0.73 \\
    Llama-3.1-8B-Instruct & default & 0.24 & 

## 9. Tests against the baselines (appendix)

Summary notebook: sections 3.2 and 3.3 (`ce.versus`, its output hidden here). Each run of `OURS_MAIN` against the baselines of its dimensionality that share our vocabulary, and against the interpretable methods; Holm over the baselines of a block.

In [25]:
BLOCKS = ['Same dimensionality', 'Interpretable methods']
base_info = baseline_rows([m for m in S.df.index if S.df.at[m, 'family'] not in ce.OURS])
interp = [m for m in base_info.index if F.at[m, 'kind'].startswith('interpretable')]
VS = {}
with capture_output():
    for m in ours_main:
        lr = int(S.df.at[m, 'rank'])
        matched = ce.matched_baselines(S, {lr: [m]}, VOCAB_MIN)[lr][1]
        same_dim = [b for b in base_info.index if b in matched]
        VS[m] = {BLOCKS[0]: ce.versus(S, m, same_dim) if same_dim else None, BLOCKS[1]: ce.versus(S, m, interp)}

KEYS = {'pol_d': 'POLAR mean Δ', 'pol_p': 'POLAR p (Holm)', 'acc_d': 'judge acc Δ', 'acc_p': 'judge acc p (Holm)'}
rows = []
for block in BLOCKS:
    found = [VS[m][block] for m in ours_main if VS[m][block] is not None]
    for b in dict.fromkeys(b for t in found for b in t['levels']):
        row = {'section': block, 'Baseline': baseline_label(b)}
        for i, m in enumerate(ours_main):
            t = VS[m][block]
            hit = t[t['levels'].eq(b)] if t is not None else []
            row.update({f'{k}{i}': hit[col].iloc[0] if len(hit) else np.nan for k, col in KEYS.items()})
        rows.append(row)
vs = pd.DataFrame(rows)
data = vs.drop(columns='section')
fmt = {'Baseline': raw, **{c: fmt_num(2, sign=True) for c in data if c[:5] in ('pol_d', 'acc_d')},
       **{c: fmt_p for c in data if c[:5] in ('pol_p', 'acc_p')}}
significant = pd.DataFrame({c: data[c].lt(ce.ALPHA) for c in data if c[:5] in ('pol_p', 'acc_p')})
body = cells(data, fmt, bold=significant)
emit('baseline_tests', latex_table(
    body, label='tab:baseline-tests', sections=vs['section'], wide=True,
    header={c: r'$\Delta$' if c[4] == 'd' else '$p$' for c in data if c != 'Baseline'},
    groups=[[('', 1), *[(rf"{run_title(S.df.loc[m])}, $r{{=}}{int(S.df.at[m, 'rank'])}$", 4) for m in ours_main]],
            [('', 1), *[('Downstream', 2), ('Word intrusion', 2)] * len(ours_main)]],
    caption=(r'Our models against each baseline, on our vocabulary. $\Delta$ = ours $-$ baseline: the mean '
             r'difference over the ' + str(len(T)) + r' downstream tasks, and the difference in word-intrusion '
             r'accuracy. Downstream: Wilcoxon signed-rank test over the tasks; word intrusion: $\chi^2$ test over '
             r"the judge's trials. All $p$ two-sided, Holm-corrected over the baselines of a block; bold: "
             r'$p < 0.05$.')))

\begin{table*}[t]
  \centering
  \small
  \begin{tabular}{lrrrrrrrr}
    \toprule
     & \multicolumn{4}{c}{Tucker, 3-gram, $r{=}300$} & \multicolumn{4}{c}{Tucker-TT, 4-gram, $r{=}300$} \\
    \cmidrule(lr){2-5} \cmidrule(lr){6-9}
     & \multicolumn{2}{c}{Downstream} & \multicolumn{2}{c}{Word intrusion} & \multicolumn{2}{c}{Downstream} & \multicolumn{2}{c}{Word intrusion} \\
    \cmidrule(lr){2-3} \cmidrule(lr){4-5} \cmidrule(lr){6-7} \cmidrule(lr){8-9}
    Baseline & $\Delta$ & $p$ & $\Delta$ & $p$ & $\Delta$ & $p$ & $\Delta$ & $p$ \\
    \midrule
    \multicolumn{9}{l}{\textit{Same dimensionality}} \\
    GloVe$_{300}$ & $-0.05$ & 0.391 & $+0.51$ & \textbf{$<$0.001} & $-0.05$ & 0.625 & $+0.49$ & \textbf{$<$0.001} \\
    NNSE$_{300}$ & $+0.00$ & 1.000 & $+0.05$ & 0.110 & $+0.00$ & 0.945 & $+0.02$ & 0.508 \\
    \midrule
    \multicolumn{9}{l}{\textit{Interpretable methods}} \\
    POLAR (GloVe) & $-0.03$ & 1.000 & $+0.31$ & \textbf{$<$0.001} & $-0.03$ & 1.000 & $+0.28$ & \textbf{$<$0

## Files

Every table above, once more in one file (`tables/all_tables.tex`), in the order of this notebook.

In [26]:
(OUT_DIR / 'all_tables.tex').write_text('\n\n'.join(TABLES.values()) + '\n', encoding='utf-8')
for name in TABLES:
    print(OUT_DIR / f'{name}.tex')
print(OUT_DIR / 'all_tables.tex')

/home/pricie/stefa/pc/5_evaluation/COLING/tables/baselines.tex
/home/pricie/stefa/pc/5_evaluation/COLING/tables/param_tests.tex
/home/pricie/stefa/pc/5_evaluation/COLING/tables/seeds.tex
/home/pricie/stefa/pc/5_evaluation/COLING/tables/iter_time.tex
/home/pricie/stefa/pc/5_evaluation/COLING/tables/grid.tex
/home/pricie/stefa/pc/5_evaluation/COLING/tables/all_runs.tex
/home/pricie/stefa/pc/5_evaluation/COLING/tables/intrusion.tex
/home/pricie/stefa/pc/5_evaluation/COLING/tables/judges.tex
/home/pricie/stefa/pc/5_evaluation/COLING/tables/baseline_tests.tex
/home/pricie/stefa/pc/5_evaluation/COLING/tables/all_tables.tex
